In [1]:
import os
import numpy as np
import pandas as pd

# Chemical reading files
cassava_chemical_readings = '../spectral_data/cassava-scores and rt-pcr.xlsx' 
maize_chemical_readings = '../spectral_data/Maize Scores and rtpcr.xlsx' 
general_expert_scores = '../spectral_data/ICAIN Disease data.xlsx' 


# Reading files
cmd_df = pd.read_excel(cassava_chemical_readings, sheet_name='CMD')
cbb_df = pd.read_excel(cassava_chemical_readings, sheet_name= 'CBB')

In [2]:
week_col = 'CBSD'
week_rows = cmd_df.astype(str).apply(
    lambda row: row.str.contains(r'WEEK\s*\d+', case=False, na=False).any(),
    axis=1
)

cmd_df['week'] = (
    cmd_df[week_col]
    .astype(str)
    .str.extract(r'WEEK\s*(\d+)', expand=False)
)
# apply a forward fill to the vales
cmd_df['week'] = cmd_df['week'].ffill()

# remove rows that contain WEEK labels
cmd_df = cmd_df[
    ~cmd_df[week_col]
    .astype(str)
    .str.contains(r'WEEK', case=False, na=False)
]

# reset index
cmd_df = cmd_df.reset_index(drop=True)

# format the columns correctly
new_col_names = ['plant_number', 'score', 'titer_1', 'l_1' ,'titer_2','l_2', 'titer_3', 'l3', 'week']
col_names = list(cmd_df.columns)
rename_dict = {old_col:new_col for old_col, new_col in zip(col_names, new_col_names)}
cmd_df = cmd_df.rename(columns=rename_dict)

# extracting plant number only
cmd_df['plant_number'] = (
    cmd_df['plant_number']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype(int)
)

# Replacing week with NaN with 1 to not week before innoculation
cmd_df['week'] = cmd_df['week'].fillna(1)
cmd_df['disease_class'] = 'CMD'
cmd_df

,plant_number,score,titer_1,l_1,titer_2,l_2,titer_3,l3,week,disease_class
0,1,1,undetected,N,Undetected,N,Undetected,N,1,CMD
1,2,1,undetected,N,Undetected,N,Undetected,N,1,CMD
2,3,1,36.224435,T,Undetected,N,Undetected,N,1,CMD
3,4,1,undetected,N,Undetected,N,Undetected,N,1,CMD
4,5,1,undetected,N,Undetected,N,Undetected,N,1,CMD
5,1,1,35.290593,WP,Undetected,N,36.401321,P,2,CMD
6,2,1,undetected,N,Undetected,N,36.701332,P,2,CMD
7,3,2,32.411178,P,Undetected,N,36.701332,N,2,CMD
8,4,1,34.812006,P,35.474634,P,37.301354,P,2,CMD
9,5,1,33.611592,P,35.889542,P,37.301354,N,2,CMD


In [3]:
cbb_df

# checking weeks with week anotations
week_col = 'Cassava bacterial blight'
week_mask = cbb_df.iloc[:, 0].astype(str).str.match(r'^\s*week\s*\d+', case=False, na=False)
cbb_df['week'] = cbb_df.iloc[:, 0].where(week_mask).str.extract(r'(\d+)', expand=False)
cbb_df['week'] = cbb_df['week'].ffill()
cbb_df = cbb_df[~week_mask].copy()
cbb_df = cbb_df.drop(0).reset_index(drop=True)

# format column names
new_col_names = ['plant_number', 'description', 'score' ,'titer_1', 'l_1' ,'titer_2','l_2', 'titer_3', 'l2', 'week']
col_names = list(cbb_df.columns)
rename_dict = {old_col:new_col for old_col, new_col in zip(col_names, new_col_names)}
cbb_df = cbb_df.rename(columns=rename_dict)

# extracting plant number only
cbb_df['plant_number'] = (
    cbb_df['plant_number']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype(int)
)
cbb_df['disease_class'] = 'CBB'
cbb_df = cbb_df.drop(columns= ['description'])

cbb_df.head(50)

,plant_number,score,titer_1,l_1,titer_2,l_2,titer_3,l2,week,disease_class
0,1,1,38.260261,0,38.65777,0,38.65777,NaN,1,CBB
1,2,1,39.75092,0,39.75092,0,39.75092,NaN,1,CBB
2,3,1,38.757147,0,39.75092,0,39.75092,NaN,1,CBB
3,4,1,39.75092,0,39.75092,0,39.75092,NaN,1,CBB
4,5,1,39.75092,0,39.75092,0,39.75092,NaN,1,CBB
5,1,1,38.03436,NaN,37.874078,NaN,38.60225,NaN,2,CBB
6,2,1,39.2418,NaN,39.284283,NaN,39.702509,NaN,2,CBB
7,3,1,37.7325,NaN,39.787927,NaN,38.502433,NaN,2,CBB
8,4,1,39.7449,NaN,39.284283,NaN,38.60244,NaN,2,CBB
9,5,1,40.248,NaN,39.284283,NaN,38.702446,NaN,2,CBB


In [4]:
## cleaning the maize expert file
maize_df = pd.read_excel(maize_chemical_readings)
week_mask = maize_df.iloc[:, 0].astype(str).str.match(
    r'^\s*week\s+\d+\s*$',
    case=False,
    na=False
)
maize_df['week'] = maize_df.iloc[:, 0].where(week_mask).str.extract(r'(\d+)', expand=False)
maize_df['week'] = maize_df['week'].ffill()
maize_df['week'] = maize_df['week'].fillna(1)

maize_df = maize_df.drop(columns={'week 1', 'Disease description'})

# format column names
def rename_df(df: pd.DataFrame):
    new_col_names= ['plant_number', 'score', 'titer_1', 'l_1', 'titer_2', 'l_2', 'titer_3', 'l3', \
                     'week', 'disease_class']
    col_names = list(df.columns)
    rename_dict = {old_col: new_col for old_col, new_col in zip(col_names, new_col_names)}
    return df.rename(columns= rename_dict)


# splitting the dataframe
split_idx = maize_df.columns.get_loc('Symptom description.1')

# creating and populating the mln meta data
mln_df = maize_df.iloc[:, :split_idx + 1]
mln_df['week'] = maize_df['week']
mln_df['disease_class'] = 'MLN'


#  creating and population the msv meta data
msv_df = maize_df.iloc[:, split_idx + 1:]
msv_df['DAY'] = maize_df['DAY']
msv_df['score'] = maize_df['score']
msv_df['disease_class'] = 'MSV'

# rearrange the positions for msv_df and perform renaming 
msv_df = msv_df[['DAY', 'score', 'MSV1', 'Symptom description (A= asymptomatic, S=Symptom)', 'MLN2',
       'Unnamed: 13', 'MLN3.1', 'Unnamed: 15', 'week','disease_class']]

mln_df = rename_df(mln_df)
msv_df = rename_df(msv_df)

# mln_df
msv_df

,plant_number,score,titer_1,l_1,titer_2,l_2,titer_3,l3,week,disease_class
0,1,1,38.114760,A,39.737000,A,38.07600,A,1,MSV
1,2,1,39.318384,A,38.630400,A,39.27840,A,1,MSV
2,3,1,38.415666,A,39.535800,A,38.37660,A,1,MSV
3,4,1,39.518988,A,39.938200,A,39.47880,A,1,MSV
4,5,1,39.719592,A,39.535800,A,39.67920,A,1,MSV
5,1,1,38.515968,A,39.636400,A,38.47680,A,2,MSV
6,2,1,39.518988,A,37.926200,A,39.47880,A,2,MSV
7,3,1,38.315364,A,38.026800,A,38.27640,A,2,MSV
8,4,1,38.515968,A,38.529800,A,38.47680,A,2,MSV
9,5,1,38.515968,A,38.630400,A,38.47680,A,2,MSV


In [5]:
mln_df

,plant_number,score,titer_1,l_1,titer_2,l_2,titer_3,l3,week,disease_class
0,1,1,38.3146,A,39.021936,A,39.720268,A,1,MLN
1,2,1,40.1200,A,40.228800,A,38.712139,A,1,MLN
2,3,1,39.5182,A,39.122508,A,39.518642,A,1,MLN
3,4,1,40.1200,A,39.625368,A,38.712139,A,1,MLN
4,5,1,40.1200,A,39.625368,A,38.712139,A,1,MLN
5,1,1,38.2143,A,38.317932,A,38.712139,A,2,MLN
6,2,1,40.1200,A,40.228800,A,39.720268,A,2,MLN
7,3,1,40.1200,A,40.228800,A,38.510513,A,2,MLN
8,4,1,40.1200,A,40.228800,A,40.325145,A,2,MLN
9,5,1,39.2173,A,39.826512,A,40.325145,A,2,MLN
